# Measures and Aggregations

In SLayer, a **measure** is a named row-level SQL expression. The **aggregation** (sum, avg, count, etc.) is specified at query time using **colon syntax**: `revenue:sum`, `*:count`, `price:weighted_avg(weight=quantity)`.

This notebook demonstrates the full range of aggregation features with working code.

**Prerequisites:** `pip install motley-slayer[examples]` (jafgen is installed by the cell below)

In [1]:
# Install jafgen (Jaffle Shop data generator) from a specific commit
# The released PyPI version has a bug; this pinned commit is the fix.
# This is only needed for running the tutorials — not a SLayer dependency.
!pip install -q git+https://github.com/rossbowen/jaffle-shop-generator.git@09557a1118b000071f8171aa97d54d5029bf0f0b


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
import sys

sys.path.insert(0, os.path.join(os.getcwd(), "..", "..", ".."))
sys.path.insert(0, os.path.join(os.getcwd(), "..", "jaffle_data"))

from setup_jaffle import ensure_jaffle_shop

engine, storage, models = ensure_jaffle_shop()

## Basic Colon Syntax

Every field formula specifies both the measure and the aggregation, separated by a colon. `*:count` is the universal row count.

In [3]:
result = engine.execute_sync(
    query={
        "source_model": "orders",
        "measures": ["*:count", "order_total:sum", "order_total:avg", "order_total:min", "order_total:max"],
    }
)

row = result.data[0]
print(f"Orders:  {row['orders._count']:,}")
print(f"Total:   ${row['orders.order_total_sum']:,.2f}")
print(f"Average: ${row['orders.order_total_avg']:.2f}")
print(f"Min:     ${row['orders.order_total_min']:.2f}")
print(f"Max:     ${row['orders.order_total_max']:.2f}")

Orders:  206,136
Total:   $2,682,434.96
Average: $13.01
Min:     $0.00
Max:     $121.47


## Multiple Aggregations on One Measure

One measure, many aggregations — no need to define `order_total_sum`, `order_total_avg`, etc. as separate measures.

In [4]:
result = engine.execute_sync(
    query={
        "source_model": "orders",
        "measures": ["*:count", "order_total:sum", "order_total:avg"],
        "dimensions": ["stores.name"],
        "order": [{"column": "order_total_sum", "direction": "desc"}],
        "limit": 5,
    }
)

print(f"{'Store':<20} {'Orders':>8} {'Total':>14} {'Average':>10}")
print("-" * 54)
for row in result.data:
    print(
        f"{row['orders.stores.name']:<20} {row['orders._count']:>8,} ${row['orders.order_total_sum']:>13,.2f} ${row['orders.order_total_avg']:>9.2f}"
    )

Store                  Orders          Total    Average
------------------------------------------------------
Brooklyn               83,419 $ 1,068,207.08 $    12.81
Philadelphia           59,138 $   766,555.51 $    12.96
Chicago                31,722 $   422,840.31 $    13.33
San Francisco          27,742 $   370,138.50 $    13.34
New Orleans             4,115 $    54,693.56 $    13.29


## Arithmetic on Aggregated Measures

Aggregated measures compose naturally in arithmetic expressions.

In [5]:
result = engine.execute_sync(
    query={
        "source_model": "orders",
        "measures": [
            "*:count",
            "order_total:sum",
            {"formula": "order_total:sum / *:count", "name": "aov", "label": "Avg Order Value"},
        ],
        "dimensions": ["stores.name"],
        "order": [{"column": "order_total_sum", "direction": "desc"}],
        "limit": 5,
    }
)

for row in result.data:
    print(f"{row['orders.stores.name']}: AOV = ${row['orders.aov']:.2f}")

Brooklyn: AOV = $12.81
Philadelphia: AOV = $12.96
Chicago: AOV = $13.33
San Francisco: AOV = $13.34
New Orleans: AOV = $13.29


## Transforms on Aggregated Measures

All transform functions (`cumsum`, `change`, `time_shift`, `rank`, etc.) wrap aggregated measure refs.

In [6]:
result = engine.execute_sync(
    query={
        "source_model": "orders",
        "measures": [
            "order_total:sum",
            {"formula": "cumsum(order_total:sum)", "name": "running_total"},
            {"formula": "change(order_total:sum)", "name": "mom_change"},
        ],
        "time_dimensions": [{"dimension": "ordered_at", "granularity": "month"}],
        "order": [{"column": "ordered_at", "direction": "asc"}],
        "limit": 6,
    }
)

print(f"{'Month':<12} {'Revenue':>14} {'Running Total':>14} {'MoM Change':>12}")
print("-" * 54)
for row in result.data:
    month = str(row["orders.ordered_at"])[:7]
    rev = row["orders.order_total_sum"]
    running = row["orders.running_total"]
    change = row["orders.mom_change"]
    change_str = f"${change:>11,.2f}" if change is not None else f"{'N/A':>12}"
    print(f"{month:<12} ${rev:>13,.2f} ${running:>13,.2f} {change_str}")

Month               Revenue  Running Total   MoM Change
------------------------------------------------------
2018-09      $     5,387.60 $     5,387.60          N/A
2018-10      $     9,807.56 $    15,195.16 $   4,419.96
2018-11      $    12,766.60 $    27,961.76 $   2,959.04
2018-12      $    15,394.21 $    43,355.97 $   2,627.61
2019-01      $    20,245.35 $    63,601.32 $   4,851.14
2019-02      $    18,964.09 $    82,565.41 $  -1,281.26


## Cross-Model Aggregated Measures

Cross-model measures use dot syntax for the join path, colon for the aggregation. `customers.*:count` means COUNT(*) on the joined customers model.

In [7]:
result = engine.execute_sync(
    query={
        "source_model": "orders",
        "measures": [
            "*:count",
            "order_total:sum",
            "customers.*:count",
        ],
        "time_dimensions": [{"dimension": "ordered_at", "granularity": "month"}],
        "order": [{"column": "ordered_at", "direction": "asc"}],
        "limit": 6,
    }
)

print(f"{'Month':<12} {'Orders':>8} {'Revenue':>14} {'Customers':>10}")
print("-" * 46)
for row in result.data:
    month = str(row["orders.ordered_at"])[:7]
    orders = row["orders._count"]
    rev = row["orders.order_total_sum"]
    custs = row["orders.customers._count"]
    print(f"{month:<12} {orders:>8,} ${rev:>13,.2f} {custs:>10,}")

Month          Orders        Revenue  Customers
----------------------------------------------
2018-09           401 $     5,387.60        401
2018-10           746 $     9,807.56        746
2018-11           951 $    12,766.60        951
2018-12         1,215 $    15,394.21      1,215
2019-01         1,563 $    20,245.35      1,563
2019-02         1,445 $    18,964.09      1,445


## Inspecting Model Measures

Auto-ingested models now have one measure per column — not five.

In [ ]:
# In v2 every column carries its data type and (optionally) an explicit
# `allowed_aggregations` whitelist. There's no separate row-level measures
# list — every column can be aggregated at query time via colon syntax.
orders_model = next(m for m in models if m.name == "orders")

print(f"orders_model has {len(orders_model.columns)} columns:")
for col in orders_model.columns:
    sql_str = col.sql or col.name
    print(f"  {col.name:<20} sql: {sql_str:<20} type: {col.type}")

print("\nAny column can be aggregated at query time via colon syntax,")
print("subject to the column's type and allowed_aggregations whitelist.")
print("Primary keys default to count / count_distinct only.")
print("Plus *:count is always available for COUNT(*).")


## Custom Aggregation: Weighted Average

Let's define a model with a custom weighted average aggregation.

In [ ]:
# Define a custom aggregation alongside built-ins. v2 unifies row-level
# definitions into Column; the custom Aggregation lives on the model.
from slayer.core.enums import DataType
from slayer.core.models import Aggregation, AggregationParam, Column, SlayerModel

items_model = SlayerModel(
    name="order_items_custom",
    sql_table="order_items",
    data_source="jaffle_shop",
    columns=[
        Column(name="sku", sql="sku", type=DataType.STRING),
        Column(name="quantity", sql="quantity", type=DataType.NUMBER),
        Column(name="price", sql="price", type=DataType.NUMBER),
    ],
    aggregations=[
        Aggregation(
            name="weighted_avg_price",
            formula="SUM({value} * {weight}) / NULLIF(SUM({weight}), 0)",
            params=[AggregationParam(name="weight", sql="quantity")],
            description="Quantity-weighted average price",
        ),
    ],
)
print(f"Custom aggregations on items_model: {[a.name for a in items_model.aggregations]}")


## Allowed Aggregations Whitelist

Restrict which aggregations make sense for a given measure.

In [ ]:
# Restrict per-column aggregation eligibility via `allowed_aggregations`. The
# whitelist overrides the type-default eligibility rule (numeric columns get
# sum/avg/min/max/etc by default; PK columns are always count-only).
from slayer.core.enums import DataType
from slayer.core.models import Column, SlayerModel

restricted_model = SlayerModel(
    name="restricted_items",
    sql_table="order_items",
    data_source="jaffle_shop",
    columns=[
        Column(name="sku", sql="sku", type=DataType.STRING),
        Column(
            name="quantity", sql="quantity", type=DataType.NUMBER,
            allowed_aggregations=["sum", "avg"],
        ),
    ],
)
qty_col = next(c for c in restricted_model.columns if c.name == "quantity")
print(f"quantity column allows: {qty_col.allowed_aggregations}")


## Generated SQL

The colon syntax compiles to standard SQL aggregation functions.

In [11]:
result = engine.execute_sync(
    query={
        "source_model": "orders",
        "measures": ["*:count", "order_total:sum", "order_total:avg"],
        "dimensions": ["stores.name"],
        "limit": 3,
    }
)

print("Generated SQL:")
print(result.sql)

Generated SQL:
SELECT
  stores.name AS "orders.stores.name",
  COUNT(*) AS "orders._count",
  SUM(orders.order_total) AS "orders.order_total_sum",
  AVG(orders.order_total) AS "orders.order_total_avg"
FROM orders AS orders
LEFT JOIN stores AS stores ON orders.store_id = stores.id
GROUP BY
  stores.name
LIMIT 3


## Summary

| Concept | Syntax | Example |
|---------|--------|---------|
| Aggregation at query time | `measure:agg` | `revenue:sum`, `price:avg` |
| Row count | `*:count` | Always available, no measure needed |
| Non-null count | `col:count` | `email:count` |
| Distinct count | `col:count_distinct` | `customer_id:count_distinct` |
| First/last by time | `col:first`, `col:last` | `balance:last(updated_at)` |
| Weighted average | `col:weighted_avg(weight=w)` | `price:weighted_avg(weight=qty)` |
| Percentile | `col:percentile(p=0.95)` | `latency:percentile(p=0.99)` |
| Median | `col:median` | Shorthand for percentile(p=0.5) |
| Custom formula | Define in model `aggregations` | `score:trimmed_mean(lo=10, hi=90)` |
| Whitelist | `allowed_aggregations` on measure | Restricts valid aggregations |

See the [aggregations guide](aggregations.md) for the full explanation.